# Let's Build a Vector Database from Scratch

> *The best way to understand something is to build it yourself.*

A vector database sounds intimidating. ChromaDB, Pinecone, Weaviate, Qdrant -- big names, complex docs.

Here is what they actually do: **take a sentence, turn it into a list of numbers, store it, and find the most similar lists when you search.**

We are going to build one from scratch, then progressively upgrade it. By the end we have four versions and a benchmark that shows exactly what each layer of engineering buys you:

1. **VectorDB** -- brute-force cosine similarity over SQLite BLOBs (60 lines of Python)
2. **FAISS standalone** -- pure C++ vector index, no database
3. **FaissVectorDB** -- FAISS index wrapped inside our SQLite database (best of both)
4. **ChromaDB** -- a real production database doing the same thing with more polish

No magic. Just NumPy, SQLite (built into Python), FAISS, and Ollama.

---
### What you need
- Ollama running locally (`ollama serve`) with `nomic-embed-text` pulled
- `uv pip install numpy faiss-cpu chromadb requests` (or see pyproject.toml)

---
## Part 1 -- What Is an Embedding?

When you call an embedding model on a sentence, you get back a list of floating-point numbers. Nothing more.

Let's call Ollama and look at what actually comes back before doing anything else:

In [25]:
import numpy as np
import requests

def get_embedding(text: str, model: str = "nomic-embed-text") -> np.ndarray:
    """One HTTP call to Ollama. Returns a 768-D float32 numpy array."""
    resp = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": model, "input": text},
        timeout=30,
    )
    resp.raise_for_status()
    return np.array(resp.json()["embeddings"][0], dtype=np.float32)

# Call it on one sentence and look at the raw output
vec = get_embedding("The cat sat on the mat")

print(f"Type  : {type(vec)}")
print(f"Shape : {vec.shape}")      # 768 numbers
print(f"Dtype : {vec.dtype}")      # 32-bit floats
print(f"Min   : {vec.min():.4f}")
print(f"Max   : {vec.max():.4f}")
print()
print("First 20 values:")
print(np.round(vec[:20], 4))

Type  : <class 'numpy.ndarray'>
Shape : (768,)
Dtype : float32
Min   : -0.1314
Max   : 0.1065

First 20 values:
[ 0.0309  0.0397 -0.1314 -0.0121  0.0568  0.0491  0.004   0.0175 -0.0613
 -0.0749 -0.0133  0.0741  0.0762  0.0166 -0.007  -0.0466 -0.005  -0.0518
  0.0044 -0.0071]


Those 768 numbers represent the *meaning* of the sentence. No single number means anything alone -- the meaning is the full pattern across all 768.

Here is the key test: similar sentences should produce similar vectors, even with completely different words:

In [26]:
sentences = [
    "The cat sat on the mat",          # baseline
    "A kitten rested on the rug",      # same meaning, ZERO shared words
    "The dog played in the park",      # related (animal) but different
    "The stock market crashed today",  # totally unrelated
]

vecs = [get_embedding(s) for s in sentences]
for s, v in zip(sentences, vecs):
    print(f"  shape={v.shape}  ->  '{s}' ")

  shape=(768,)  ->  'The cat sat on the mat' 
  shape=(768,)  ->  'A kitten rested on the rug' 
  shape=(768,)  ->  'The dog played in the park' 
  shape=(768,)  ->  'The stock market crashed today' 


---
## Part 2 -- The Math: Cosine Similarity

Each embedding is a point in 768-dimensional space. 'Similar meaning' means 'nearby points.'

We measure closeness using **cosine similarity** -- the cosine of the angle between two vectors.

The formula:
```
cosine(A, B) = dot(A, B) / (norm(A) * norm(B))
```

- **+1** -> same direction -> same meaning  
- **0** -> perpendicular -> unrelated  
- **-1** -> opposite -> contradictory meaning

Let's write it from scratch -- no scipy, no sklearn:

In [27]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity written out explicitly.

    a, b: shape (D,)  -- two embedding vectors
    returns: float in [-1, 1]
    """
    dot    = np.dot(a, b)           # step 1: sum of element-wise products
    norm_a = np.linalg.norm(a)      # step 2: sqrt(sum of squares)
    norm_b = np.linalg.norm(b)      # step 3: same for b
    return float(dot / (norm_a * norm_b))

# Compare every sentence against the first one
baseline = vecs[0]
print(f'Baseline: "{sentences[0]}"')
print()
for s, v in zip(sentences[1:], vecs[1:]):
    sim = cosine_similarity(baseline, v)
    bar = "=" * int(sim * 40)
    print(f"  {sim:.4f}  {bar}")
    print(f"           '{s}'")
    print()

Baseline: "The cat sat on the mat"

  0.7050  ============================
           'A kitten rested on the rug'

  0.4996  ===================
           'The dog played in the park'

  0.3932  ===============
           'The stock market crashed today'



The 'kitten' sentence scores **much higher** than the 'stock market' sentence -- despite sharing zero words with the baseline.

The model learned: cat ~ kitten, sat ~ rested, mat ~ rug.

**This one operation -- cosine similarity -- is the core engine inside every vector database. Everything else is storage and engineering on top.**

---
## Part 3 -- Building the Database

Now we know what embeddings are and how to compare them. Let's build the database.

We need three things:
1. **Store** -- save the text, its vector, and any metadata
2. **Search** -- given a query vector, find the most similar stored ones
3. **Persist** -- survive a Python restart (data on disk, not just RAM)

The simplest tool that does all three: **SQLite** -- built into Python, zero setup, single file on disk.

Here is our schema:
```sql
CREATE TABLE documents (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    text     TEXT,        -- the original sentence
    metadata TEXT,        -- JSON: topic, source, date, etc.
    vector   BLOB         -- raw float32 bytes: vec.tobytes()
)
```

Vectors are stored as binary blobs. `np.ndarray.tobytes()` serializes. `np.frombuffer()` deserializes. Simple.

Here is the whole class -- 60 lines:

In [28]:
import sqlite3
import json

class VectorDB:
    """
    A vector database built from scratch.

    Storage : SQLite  -- one .db file holds everything
    Search  : NumPy   -- load all vectors, cosine similarity, return top-k
    Persist : free    -- pass a file path, data survives process restarts
    """

    def __init__(self, path: str = ":memory:"):
        self.path = path
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id       INTEGER PRIMARY KEY AUTOINCREMENT,
                text     TEXT    NOT NULL,
                metadata TEXT    NOT NULL DEFAULT '{}',
                vector   BLOB    NOT NULL
            )
        """)
        self.conn.commit()

    # --- Write ---------------------------------------------------------------

    def add(self, text: str, vector: np.ndarray, metadata: dict = None) -> int:
        """Insert one document. Returns its row ID."""
        cur = self.conn.execute(
            "INSERT INTO documents (text, metadata, vector) VALUES (?, ?, ?)",
            (text, json.dumps(metadata or {}), vector.astype(np.float32).tobytes()),
        )
        self.conn.commit()
        return cur.lastrowid

    def add_batch(self, texts: list, vectors: np.ndarray, metadatas: list = None):
        """Insert many in one SQL call -- far faster than looping add()."""
        metadatas = metadatas or [{} for _ in texts]
        self.conn.executemany(
            "INSERT INTO documents (text, metadata, vector) VALUES (?, ?, ?)",
            [
                (t, json.dumps(m), np.asarray(v, dtype=np.float32).tobytes())
                for t, m, v in zip(texts, metadatas, vectors)
            ],
        )
        self.conn.commit()

    # --- Read ----------------------------------------------------------------

    def search(self, query: np.ndarray, top_k: int = 5) -> list:
        """
        Find the top_k most similar documents.

        Returns: list of (text, score, metadata) sorted by similarity descending.

        What happens:
          1. SELECT all rows from SQLite
          2. Deserialize each BLOB -> numpy float32 array
          3. Stack into matrix M of shape (N, 768)
          4. Cosine similarity: (M @ q) / (|M| * |q|)
          5. argsort descending, return top_k
        """
        rows = self.conn.execute(
            "SELECT text, metadata, vector FROM documents"
        ).fetchall()
        if not rows:
            return []

        q = query.astype(np.float32)
        texts_out, metas, vecs = [], [], []
        for text, meta_json, blob in rows:
            texts_out.append(text)
            metas.append(json.loads(meta_json))
            vecs.append(np.frombuffer(blob, dtype=np.float32))

        M    = np.stack(vecs)                                    # (N, 768)
        sims = (M @ q) / (np.linalg.norm(M, axis=1) * np.linalg.norm(q) + 1e-10)
        top  = np.argsort(-sims)[:top_k]
        return [(texts_out[i], float(sims[i]), metas[i]) for i in top]

    @property
    def count(self) -> int:
        return self.conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]

    def __repr__(self) -> str:
        return f"VectorDB(path={self.path!r}, docs={self.count})"

# Quick sanity check
db = VectorDB()
print(db)

VectorDB(path=':memory:', docs=0)


In [29]:
# Add one document and immediately search it
test_vec = get_embedding("Cats are curious and independent creatures")
db.add("Cats are curious and independent creatures", test_vec, metadata={"topic": "animals"})

q_vec   = get_embedding("What pets like to explore on their own?")
results = db.search(q_vec, top_k=1)

for text, score, meta in results:
    print(f"score : {score:.4f}")
    print(f"text  : {text}")
    print(f"meta  : {meta}")

score : 0.7148
text  : Cats are curious and independent creatures
meta  : {'topic': 'animals'}


---
## Part 4 -- Demo: Insert Real Documents and Search

Let's load 50 real sentences across 5 topics. These same documents will be used for the benchmark -- no random vectors, real embeddings.

*This takes 30-60 seconds to embed. Ollama makes one API call per sentence.*

In [30]:
DOCUMENTS = [
    # Science
    ("Photosynthesis converts CO2 and water into glucose using sunlight.",                {"topic": "science"}),
    ("DNA encodes genetic instructions using four bases: adenine, thymine, guanine, cytosine.", {"topic": "science"}),
    ("The speed of light in a vacuum is exactly 299,792,458 metres per second.",          {"topic": "science"}),
    ("Black holes form when massive stars collapse under their own gravitational pull.",  {"topic": "science"}),
    ("Neurons transmit signals via electrochemical impulses across synaptic gaps.",       {"topic": "science"}),
    ("The periodic table organises elements by atomic number and chemical properties.",  {"topic": "science"}),
    ("Quantum entanglement links particles so measuring one affects the other instantly.",{"topic": "science"}),
    ("CRISPR-Cas9 lets scientists cut and edit DNA sequences with high precision.",       {"topic": "science"}),
    ("Plate tectonics explains the movement of Earth's lithosphere over millions of years.", {"topic": "science"}),
    ("Mitochondria generate ATP through cellular respiration, powering the cell.",        {"topic": "science"}),
    # History
    ("The Roman Empire fell in 476 AD when Romulus Augustulus was deposed.",             {"topic": "history"}),
    ("The Black Death killed roughly a third of Europe's population in the 14th century.", {"topic": "history"}),
    ("Gutenberg's printing press revolutionised information sharing around 1440.",        {"topic": "history"}),
    ("The French Revolution began in 1789 with the storming of the Bastille.",           {"topic": "history"}),
    ("World War II ended in 1945 after atomic bombs fell on Hiroshima and Nagasaki.",    {"topic": "history"}),
    ("Ancient Egyptians built the pyramids as tombs for pharaohs over 4,500 years ago.",{"topic": "history"}),
    ("Columbus reached the Americas in 1492, opening sustained contact with Europe.",    {"topic": "history"}),
    ("The Great Wall of China stretches over 13,000 miles and took centuries to build.", {"topic": "history"}),
    ("The Russian Revolution of 1917 overthrew the Tsar and led to the Soviet Union.",  {"topic": "history"}),
    ("Cleopatra ruled Egypt as its last active pharaoh before Roman conquest.",          {"topic": "history"}),
    # Food
    ("The Maillard reaction creates browned, flavourful crusts when proteins and sugars are heated.", {"topic": "food"}),
    ("Sourdough bread uses wild yeast and lactic acid bacteria for its tangy taste.",    {"topic": "food"}),
    ("Olive oil is a cornerstone of Mediterranean cuisine and rich in monounsaturated fats.", {"topic": "food"}),
    ("Umami is the fifth basic taste, found in soy sauce, mushrooms, and aged cheese.", {"topic": "food"}),
    ("Fermentation transforms sugars into alcohol or acids, preserving food and adding depth.", {"topic": "food"}),
    ("Resting meat after cooking lets juices redistribute, making each bite more tender.", {"topic": "food"}),
    ("Saffron, from crocus flowers, is the world's most expensive spice by weight.",     {"topic": "food"}),
    ("Emulsification combines oil and water using an emulsifier like egg yolk or mustard.", {"topic": "food"}),
    ("Slow-cooking tough cuts like brisket at low heat converts collagen into gelatin.", {"topic": "food"}),
    ("The Scoville scale measures chili heat by capsaicin concentration.",               {"topic": "food"}),
    # Technology
    ("The internet evolved from ARPANET, a US military research network of the 1960s.", {"topic": "technology"}),
    ("Machine learning models learn patterns from data instead of following explicit rules.", {"topic": "technology"}),
    ("A transistor switches electrical signals and is the fundamental unit of computers.", {"topic": "technology"}),
    ("TCP/IP is the protocol suite that enables devices on different networks to communicate.", {"topic": "technology"}),
    ("Encryption converts readable data into ciphertext using mathematical algorithms.", {"topic": "technology"}),
    ("Moore's Law predicted transistor counts on chips would double every two years.",   {"topic": "technology"}),
    ("Transformers use self-attention to process sequences, underpinning modern LLMs.", {"topic": "technology"}),
    ("Open source software lets anyone view, modify, and distribute the source code.",  {"topic": "technology"}),
    ("Quantum computers use qubits in superposition, enabling exponential parallelism.", {"topic": "technology"}),
    ("A blockchain is a distributed ledger that records transactions across many nodes.", {"topic": "technology"}),
    # Sports
    ("A marathon is 42.195 km, traced to Pheidippides running from Marathon to Athens.", {"topic": "sports"}),
    ("In tennis, a player must win six games and lead by two to win a set.",             {"topic": "sports"}),
    ("The Tour de France covers roughly 3,500 km across France over three weeks.",       {"topic": "sports"}),
    ("Michael Jordan won six NBA championships with the Chicago Bulls.",                 {"topic": "sports"}),
    ("The offside rule in football stops attackers from loitering near the opponent's goal.", {"topic": "sports"}),
    ("Butterfly stroke requires simultaneous arm movement and an undulating dolphin kick.", {"topic": "sports"}),
    ("The Olympic decathlon has ten events across two days, testing all-round athleticism.", {"topic": "sports"}),
    ("Roger Federer held the world tennis number-one ranking for a record 310 weeks.",   {"topic": "sports"}),
    ("Nadia Comaneci was the first gymnast to score a perfect 10.0, at the 1976 Olympics.", {"topic": "sports"}),
    ("A cheetah can accelerate from 0 to 100 km/h in under three seconds.",             {"topic": "sports"}),
]

texts = [d[0] for d in DOCUMENTS]
metas = [d[1] for d in DOCUMENTS]

print(f"Embedding {len(texts)} sentences with Ollama...")
import time
t0 = time.perf_counter()
embeddings = [get_embedding(t) for t in texts]
elapsed = time.perf_counter() - t0

embed_matrix = np.stack(embeddings)    # (50, 768)
print(f"Done in {elapsed:.1f}s  ({elapsed/len(texts)*1000:.0f} ms/sentence)")
print(f"Matrix shape: {embed_matrix.shape}  ({embed_matrix.nbytes/1024:.0f} KB in RAM)")

Embedding 50 sentences with Ollama...
Done in 1.2s  (24 ms/sentence)
Matrix shape: (50, 768)  (150 KB in RAM)


In [31]:
# Insert all 50 into a fresh database
db = VectorDB()
db.add_batch(texts, embed_matrix, metas)
print(db)

VectorDB(path=':memory:', docs=50)


In [32]:
# 5 queries -- none share exact words with the best-matching documents
QUERIES = [
    "How do living cells produce energy?",
    "What caused large ancient empires to collapse?",
    "What makes bread dough ferment and develop flavour?",
    "How do modern AI systems learn from data?",
    "Which athletic competitions test overall physical ability?",
]

for query in QUERIES:
    q_vec   = get_embedding(query)
    results = db.search(q_vec, top_k=3)
    print(f'Query: "{query}"')
    print("-" * 70)
    for text, score, meta in results:
        print(f"  [{meta['topic']:<10}]  {score:.4f}  {text[:65]}...")
    print()

Query: "How do living cells produce energy?"
----------------------------------------------------------------------
  [science   ]  0.7588  Mitochondria generate ATP through cellular respiration, powering ...
  [science   ]  0.5987  Photosynthesis converts CO2 and water into glucose using sunlight...
  [food      ]  0.5330  Fermentation transforms sugars into alcohol or acids, preserving ...

Query: "What caused large ancient empires to collapse?"
----------------------------------------------------------------------
  [science   ]  0.6470  Black holes form when massive stars collapse under their own grav...
  [history   ]  0.6221  Ancient Egyptians built the pyramids as tombs for pharaohs over 4...
  [history   ]  0.6134  World War II ended in 1945 after atomic bombs fell on Hiroshima a...

Query: "What makes bread dough ferment and develop flavour?"
----------------------------------------------------------------------
  [food      ]  0.7830  Sourdough bread uses wild yeast and lacti

---
## Part 5 -- Persistence: The Database Survives a Restart

SQLite handles persistence for free. Pass a file path instead of `:memory:` and all data is written to disk automatically -- no `.save()` call, no special method.

Let's prove it:

In [33]:
import os, shutil

DB_PATH = "./knowledge.db"
if os.path.exists(DB_PATH): os.remove(DB_PATH)

# Create on disk with the same 50 documents
db_disk = VectorDB(DB_PATH)
db_disk.add_batch(texts, embed_matrix, metas)
print(f"Created  : {db_disk}")
print(f"File size: {os.path.getsize(DB_PATH):,} bytes  ({os.path.getsize(DB_PATH)/1024:.1f} KB)")

# Simulate a process restart -- delete the Python object
del db_disk
print()
print("Python object deleted. Data now lives ONLY on disk.")
print()

# Reload from disk -- just point to the same file
db_reloaded = VectorDB(DB_PATH)
print(f"Reloaded : {db_reloaded}")

# Search still works
q_vec   = get_embedding("How do cells produce energy?")
results = db_reloaded.search(q_vec, top_k=3)
print()
print("Search after reload:")
for text, score, _ in results:
    print(f"  {score:.4f}  {text[:65]}...")

Created  : VectorDB(path='./knowledge.db', docs=50)
File size: 217,088 bytes  (212.0 KB)

Python object deleted. Data now lives ONLY on disk.

Reloaded : VectorDB(path='./knowledge.db', docs=50)

Search after reload:
  0.7870  Mitochondria generate ATP through cellular respiration, powering ...
  0.6199  Photosynthesis converts CO2 and water into glucose using sunlight...
  0.5563  Fermentation transforms sugars into alcohol or acids, preserving ...


In [34]:
# Peek inside the SQLite file -- everything is stored as plain SQL rows
conn = sqlite3.connect(DB_PATH)

print("Schema:")
for row in conn.execute("SELECT sql FROM sqlite_master WHERE type='table'"):
    print(" ", row[0])

print()
print("First 3 rows:")
for row in conn.execute("SELECT id, text, metadata, length(vector) FROM documents LIMIT 3"):
    doc_id, text, meta, vec_bytes = row
    print(f"  id={doc_id}  vec_size={vec_bytes} bytes  meta={meta}")
    print(f"    '{text[:60]}...'")

conn.close()
print()
print(f"Each BLOB = {768 * 4} bytes = 768 floats x 4 bytes (float32)")

Schema:
  CREATE TABLE documents (
                id       INTEGER PRIMARY KEY AUTOINCREMENT,
                text     TEXT    NOT NULL,
                metadata TEXT    NOT NULL DEFAULT '{}',
                vector   BLOB    NOT NULL
            )
  CREATE TABLE sqlite_sequence(name,seq)

First 3 rows:
  id=1  vec_size=3072 bytes  meta={"topic": "science"}
    'Photosynthesis converts CO2 and water into glucose using sun...'
  id=2  vec_size=3072 bytes  meta={"topic": "science"}
    'DNA encodes genetic instructions using four bases: adenine, ...'
  id=3  vec_size=3072 bytes  meta={"topic": "science"}
    'The speed of light in a vacuum is exactly 299,792,458 metres...'

Each BLOB = 3072 bytes = 768 floats x 4 bytes (float32)


---
## Part 6 -- Where It Gets Slow: The O(N) Cost

Our `search()` loads **every row** from SQLite, deserializes every BLOB, then does one big matrix multiply.

As N grows, every search touches every stored vector. That is O(N) -- linear time. Let's measure how it scales:

In [35]:
# Scale test with synthetic vectors (no Ollama -- just measuring DB overhead)
D        = 768
rng      = np.random.default_rng(42)
N_QUERIES = 20

print(f"{'N docs':>8}  {'ms/query':>10}  {'QPM':>10}  Notes")
print("-" * 55)

for N in [50, 500, 5_000, 50_000]:
    bench_db   = VectorDB()
    bench_vecs = rng.standard_normal((N, D)).astype(np.float32)
    bench_db.add_batch([f"doc_{i}" for i in range(N)], bench_vecs)

    q_vecs = rng.standard_normal((N_QUERIES, D)).astype(np.float32)
    t0 = time.perf_counter()
    for q in q_vecs:
        bench_db.search(q, top_k=5)
    avg_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    note = {50: "<- our real docs", 50_000: "<- starting to hurt"}.get(N, "")
    print(f"{N:>8,}  {avg_ms:>10.2f}  {60_000/avg_ms:>10,.0f}  {note}")

  N docs    ms/query         QPM  Notes
-------------------------------------------------------
      50        0.18     325,619  <- our real docs
     500        1.17      51,256  
   5,000       14.61       4,107  
  50,000      104.09         576  <- starting to hurt


The bottleneck is loading all vectors from SQLite on every query.

Two ways to make this faster:
1. **Cache vectors in RAM** after first load (reduces SQLite I/O overhead)
2. **Use a smarter index** that skips most comparisons -- which is what FAISS does

Let's compare.

---
## Part 7 -- FAISS: What Happens When You Remove the Database

FAISS (Meta AI Similarity Search) is a **pure vector index** -- no SQLite, no text storage, no metadata. Vectors live in RAM and the dot products run in C++ with SIMD instructions.

This answers: *how much of our search time is 'database overhead' vs 'actual math'?*

In [36]:
import faiss

# Build a FAISS index with the same 50 real embeddings
# FAISS uses L2 distance. For normalized vectors: L2^2 = 2 - 2*cos(theta)
# So: normalize vectors first, then convert L2^2 back to cosine similarity.

normed_matrix = embed_matrix / np.linalg.norm(embed_matrix, axis=1, keepdims=True)

faiss_index = faiss.IndexFlatL2(768)
faiss_index.add(normed_matrix.astype(np.float32))
print(f"FAISS index: {faiss_index.ntotal} vectors, dim=768")
print()

# Same process as before: start with a question, embed it, search.
query_text = "How do cells produce energy?"
print(f'Query: "{query_text}"')
print()

# Step 1: embed the query (same get_embedding call we always use)
q_vec  = get_embedding(query_text)
print(f"Embedded → shape {q_vec.shape}")

# Step 2: FAISS needs normalised vectors (we normalised the index too)
q_norm = q_vec / np.linalg.norm(q_vec)

# Step 3: search the FAISS index
D_sq, I = faiss_index.search(q_norm.reshape(1, -1), k=3)
cosine_scores = 1 - D_sq[0] / 2     # L2^2 -> cosine

print()
print("FAISS top-3:")
for idx, score in zip(I[0], cosine_scores):
    print(f"  {score:.4f}  {texts[idx][:65]}...")

FAISS index: 50 vectors, dim=768

Query: "How do cells produce energy?"

Embedded → shape (768,)

FAISS top-3:
  0.7870  Mitochondria generate ATP through cellular respiration, powering ...
  0.6199  Photosynthesis converts CO2 and water into glucose using sunlight...
  0.5563  Fermentation transforms sugars into alcohol or acids, preserving ...


In [37]:
# Speed comparison: our VectorDB vs FAISS on the SAME 50 real documents
#
# For a fair benchmark we query all 50 sentences against both systems.
# The process is always the same:
#   1. Take a query string     (we already have the 50 texts)
#   2. Get its embedding       (we already computed these in Part 4)
#   3. Search the index        (this is the step we're timing)
#
# We reuse the pre-computed embeddings because the embedding step is
# identical for both systems -- we only want to measure search speed.

# Prepare normalised queries for FAISS (same normalise step as the demo above)
normed_queries = np.stack([v / np.linalg.norm(v) for v in embeddings]).astype(np.float32)
N_Q = len(embeddings)   # 50 queries -- one per document

print(f"Running {N_Q} queries through each system...\n")
print("Example queries being timed:")
for t in texts[:3]:
    print(f'  "{t[:60]}..."')
print(f"  ... and {N_Q - 3} more\n")

# Our DB: query text → embedding → db.search(embedding)
t0 = time.perf_counter()
for q in embeddings:
    db.search(q, top_k=5)
our_ms = (time.perf_counter() - t0) / N_Q * 1000

# FAISS: query text → embedding → normalise → faiss_index.search(normalised)
t0 = time.perf_counter()
for q in normed_queries:
    faiss_index.search(q.reshape(1, -1), 5)
faiss_ms = (time.perf_counter() - t0) / N_Q * 1000

print(f"Our VectorDB  :  {our_ms:.3f} ms/query")
print(f"FAISS         :  {faiss_ms:.3f} ms/query")
print(f"Speedup       :  {our_ms/faiss_ms:.1f}x faster")
print()
print("FAISS is faster because:")
print("  - Vectors already in RAM (no SQLite SELECT on every query)")
print("  - No BLOB deserialization")
print("  - C++ SIMD dot products")
print()
print("FAISS loses:")
print("  - No text storage (you need an external lookup table)")
print("  - No metadata (no 'where topic=science' filtering)")
print("  - No persistence (index disappears when Python exits)")

Running 50 queries through each system...

Example queries being timed:
  "Photosynthesis converts CO2 and water into glucose using sun..."
  "DNA encodes genetic instructions using four bases: adenine, ..."
  "The speed of light in a vacuum is exactly 299,792,458 metres..."
  ... and 47 more

Our VectorDB  :  0.123 ms/query
FAISS         :  0.005 ms/query
Speedup       :  23.6x faster

FAISS is faster because:
  - Vectors already in RAM (no SQLite SELECT on every query)
  - No BLOB deserialization
  - C++ SIMD dot products

FAISS loses:
  - No text storage (you need an external lookup table)
  - No metadata (no 'where topic=science' filtering)
  - No persistence (index disappears when Python exits)


In [38]:
# Do they return the same top result?

VERIFY_QUERIES = [
    "How do living cells produce energy?",
    "What caused ancient civilisations to collapse?",
    "What makes bread dough ferment?",
    "How do neural networks learn?",
    "Which events test all-round athletic ability?",
]

print(f"{'Query':<50} {'Agree?':>6}")
print("-" * 58)
agree = 0
for q_text in VERIFY_QUERIES:
    q_vec  = get_embedding(q_text)
    our    = db.search(q_vec, top_k=1)[0][0]
    q_norm = q_vec / np.linalg.norm(q_vec)
    _, I   = faiss_index.search(q_norm.reshape(1, -1), 1)
    theirs = texts[I[0][0]]
    match  = our == theirs
    if match: agree += 1
    tag    = "YES" if match else "NO "
    print(f"  {q_text[:48]:<48}  {tag}")

print(f"\nAgreement: {agree}/{len(VERIFY_QUERIES)}")

Query                                              Agree?
----------------------------------------------------------
  How do living cells produce energy?               YES
  What caused ancient civilisations to collapse?    YES
  What makes bread dough ferment?                   YES
  How do neural networks learn?                     YES
  Which events test all-round athletic ability?     YES

Agreement: 5/5


---
## Part 8 -- The Upgrade: FAISS Inside Our Database

Here is where it gets interesting.

FAISS is fast but has no storage -- no text, no metadata, no persistence. Our VectorDB stores everything but searches slow.

**What if we combine them?** SQLite stores text and metadata. FAISS handles the vector search. Each tool does what it is best at.

This is exactly what production databases like ChromaDB do internally. Let's build our own version so we can see the mechanics:

```
+----------------------------------------------------------+
|                   FaissVectorDB                          |
|                                                          |
|  db.add(text, vector, metadata)                          |
|  +----------------------------------------------------+  |
|  | SQLite: INSERT (id, text, metadata)  ← no BLOB!    |  |
|  | FAISS:  index.add(vector)            ← fast C++    |  |
|  +----------------------------------------------------+  |
|                                                          |
|  db.search(query_vec, top_k)                             |
|  +----------------------------------------------------+  |
|  | 1. FAISS: index.search(query, k) → ids, distances  |  |
|  | 2. SQLite: SELECT text, metadata WHERE id IN (ids)  |  |
|  | 3. Convert L2 distances → cosine scores             |  |
|  | 4. Return [(text, score, metadata), ...]            |  |
|  +----------------------------------------------------+  |
+----------------------------------------------------------+
```

Notice what changed: search no longer loads ALL vectors. FAISS returns only the top-k indices, and we fetch only those rows from SQLite. The BLOB column is gone entirely.

In [39]:
class FaissVectorDB:
    """
    Our VectorDB -- upgraded with a FAISS index.

    Storage : SQLite  -- text and metadata (NO vector BLOBs)
    Search  : FAISS   -- vectors live in a C++ index in RAM
    Persist : two files -- SQLite .db for text, FAISS .index for vectors

    The key idea: split the work.
      - SQLite is great at storing/filtering text and JSON.
      - FAISS is great at finding nearest vectors fast.
      - Neither is great at the other's job.
    """

    def __init__(self, dim: int = 768, path: str = ":memory:"):
        self.dim   = dim
        self.path  = path
        self.conn  = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id       INTEGER PRIMARY KEY AUTOINCREMENT,
                text     TEXT    NOT NULL,
                metadata TEXT    NOT NULL DEFAULT '{}'
            )
        """)
        # -- notice: NO vector BLOB column! FAISS owns the vectors. --
        self.conn.commit()

        # FAISS flat index on normalised vectors (L2 on unit vectors = cosine)
        self.index = faiss.IndexFlatL2(dim)

    # --- Write ---------------------------------------------------------------

    def add(self, text: str, vector: np.ndarray, metadata: dict = None) -> int:
        """Insert one document. Vector goes to FAISS, text goes to SQLite."""
        # Normalise so L2 distance maps to cosine similarity
        v = vector.astype(np.float32)
        v = v / (np.linalg.norm(v) + 1e-10)

        cur = self.conn.execute(
            "INSERT INTO documents (text, metadata) VALUES (?, ?)",
            (text, json.dumps(metadata or {})),
        )
        self.conn.commit()
        self.index.add(v.reshape(1, -1))     # FAISS expects (1, D)
        return cur.lastrowid

    def add_batch(self, texts: list, vectors: np.ndarray, metadatas: list = None):
        """Insert many at once."""
        metadatas = metadatas or [{} for _ in texts]

        # Normalise all vectors
        M = np.stack([np.asarray(v, dtype=np.float32) for v in vectors])
        norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-10
        M_normed = M / norms

        # SQLite gets text + metadata
        self.conn.executemany(
            "INSERT INTO documents (text, metadata) VALUES (?, ?)",
            [(t, json.dumps(m)) for t, m in zip(texts, metadatas)],
        )
        self.conn.commit()

        # FAISS gets the vectors
        self.index.add(M_normed)

    # --- Read ----------------------------------------------------------------

    def search(self, query: np.ndarray, top_k: int = 5) -> list:
        """
        Find the top_k most similar documents.

        What happens:
          1. Normalise query vector
          2. FAISS: search(query, k) → top_k indices + L2² distances
          3. Convert L2² on unit vectors → cosine: cos = 1 - L2²/2
          4. SQLite: fetch ONLY those rows by id
          5. Return [(text, score, metadata), ...]

        The big win: step 2 touches only the FAISS index.
        No "SELECT * FROM documents". No BLOB deserialization.
        """
        q = query.astype(np.float32)
        q = q / (np.linalg.norm(q) + 1e-10)

        D_sq, I = self.index.search(q.reshape(1, -1), top_k)

        results = []
        for idx, dist in zip(I[0], D_sq[0]):
            if idx == -1:
                continue   # FAISS returns -1 when fewer than k results exist
            # SQLite IDs are 1-based (AUTOINCREMENT), FAISS is 0-based
            row = self.conn.execute(
                "SELECT text, metadata FROM documents WHERE id = ?",
                (int(idx) + 1,),
            ).fetchone()
            if row:
                cosine_score = float(1 - dist / 2)
                results.append((row[0], cosine_score, json.loads(row[1])))
        return results

    # --- Persist -------------------------------------------------------------

    def save(self, index_path: str):
        """
        Save to disk. Two files:
          - self.path  → SQLite file  (text + metadata)  -- already on disk
          - index_path → FAISS index  (vectors)           -- written here

        Why two files? SQLite can't efficiently store FAISS's internal data
        structures, and FAISS can't store text. Each saves in its own format.
        """
        faiss.write_index(self.index, index_path)

    @classmethod
    def load(cls, dim: int, db_path: str, index_path: str) -> "FaissVectorDB":
        """
        Load from disk. Reconnect SQLite and read back the FAISS index.

        This is a classmethod -- it creates a new FaissVectorDB object
        from the two files that save() wrote.
        """
        obj = cls(dim=dim, path=db_path)
        obj.index = faiss.read_index(index_path)
        return obj

    @property
    def count(self) -> int:
        return self.index.ntotal

    def __repr__(self) -> str:
        return f"FaissVectorDB(path={self.path!r}, docs={self.count})"


# Quick sanity check -- empty database, just like we did with VectorDB in Part 3
fdb = FaissVectorDB(dim=768)
print(fdb)

FaissVectorDB(path=':memory:', docs=0)


In [40]:
# Same flow as Part 3: add one document, immediately search for it.
# The student should see the exact same pattern: text → embed → add → query → embed → search.

# Step 1: embed a sentence
doc_text = "Cats are curious and independent creatures"
doc_vec  = get_embedding(doc_text)
print(f'Document: "{doc_text}"')
print(f"Embedded → shape {doc_vec.shape}")

# Step 2: add to our FaissVectorDB (vector goes to FAISS, text goes to SQLite)
fdb.add(doc_text, doc_vec, metadata={"topic": "animals"})
print(f"Added to FaissVectorDB → {fdb}")
print()

# Step 3: ask a question, embed it, search
query = "What pets like to explore on their own?"
q_vec = get_embedding(query)
print(f'Query:    "{query}"')
print(f"Embedded → shape {q_vec.shape}")
print()

# Step 4: search
results = fdb.search(q_vec, top_k=1)
for text, score, meta in results:
    print(f"score : {score:.4f}")
    print(f"text  : {text}")
    print(f"meta  : {meta}")

Document: "Cats are curious and independent creatures"
Embedded → shape (768,)
Added to FaissVectorDB → FaissVectorDB(path=':memory:', docs=1)

Query:    "What pets like to explore on their own?"
Embedded → shape (768,)

score : 0.7148
text  : Cats are curious and independent creatures
meta  : {'topic': 'animals'}


In [41]:
# Now load all 50 real documents -- same texts and embeddings from Part 4.
# We already embedded them earlier, so we reuse those vectors.
# (In production you'd embed at insert time; here we avoid re-calling Ollama.)

fdb = FaissVectorDB(dim=768)                     # start fresh
fdb.add_batch(texts, embed_matrix, metas)         # texts + embeddings from Part 4
print(f"Loaded {fdb.count} documents into FaissVectorDB")
print()

# Confirm it works: same query as Part 4 -- text → embed → search
query = "How do living cells produce energy?"
print(f'Query: "{query}"')
q_vec = get_embedding(query)
print(f"Embedded → shape {q_vec.shape}")
print()

results = fdb.search(q_vec, top_k=3)
for text, score, meta in results:
    print(f"  [{meta['topic']:<10}]  {score:.4f}  {text[:65]}...")

Loaded 50 documents into FaissVectorDB

Query: "How do living cells produce energy?"
Embedded → shape (768,)

  [science   ]  0.7588  Mitochondria generate ATP through cellular respiration, powering ...
  [science   ]  0.5987  Photosynthesis converts CO2 and water into glucose using sunlight...
  [food      ]  0.5330  Fermentation transforms sugars into alcohol or acids, preserving ...


In [42]:
# Same 5 queries -- does FaissVectorDB return the same results as our brute-force DB?

print("FaissVectorDB vs original VectorDB -- same queries, same 50 docs:\n")
for q_text in QUERIES:
    q_vec = get_embedding(q_text)

    # Original brute-force
    bf_results = db.search(q_vec, top_k=3)
    # FAISS-backed
    fi_results = fdb.search(q_vec, top_k=3)

    print(f'Query: "{q_text}"')
    print(f"  {'Brute-force top-1:':<22} {bf_results[0][0][:55]}...  ({bf_results[0][1]:.4f})")
    print(f"  {'FAISS-backed top-1:':<22} {fi_results[0][0][:55]}...  ({fi_results[0][1]:.4f})")
    match = bf_results[0][0] == fi_results[0][0]
    print(f"  Match: {'YES' if match else 'NO'}")
    print()

FaissVectorDB vs original VectorDB -- same queries, same 50 docs:

Query: "How do living cells produce energy?"
  Brute-force top-1:     Mitochondria generate ATP through cellular respiration,...  (0.7588)
  FAISS-backed top-1:    Mitochondria generate ATP through cellular respiration,...  (0.7588)
  Match: YES

Query: "What caused large ancient empires to collapse?"
  Brute-force top-1:     Black holes form when massive stars collapse under thei...  (0.6470)
  FAISS-backed top-1:    Black holes form when massive stars collapse under thei...  (0.6470)
  Match: YES

Query: "What makes bread dough ferment and develop flavour?"
  Brute-force top-1:     Sourdough bread uses wild yeast and lactic acid bacteri...  (0.7830)
  FAISS-backed top-1:    Sourdough bread uses wild yeast and lactic acid bacteri...  (0.7830)
  Match: YES

Query: "How do modern AI systems learn from data?"
  Brute-force top-1:     Machine learning models learn patterns from data instea...  (0.7281)
  FAISS-backed top-1

### Persistence: Two Files Instead of One

Our original VectorDB persisted for free -- everything was in one SQLite file. The FaissVectorDB splits storage into two systems, so persistence needs two files:

| File | Stores | Written by |
|------|--------|------------|
| `knowledge_faiss.db` | text + metadata | SQLite (automatic when `path` is a file) |
| `knowledge_faiss.index` | vectors | `faiss.write_index()` (our `save()` method) |

Let's prove it survives a restart, just like we did in Part 5:

In [ ]:
FAISS_DB_PATH    = "./knowledge_faiss.db"
FAISS_INDEX_PATH = "./knowledge_faiss.index"

# Clean up any previous run
for p in [FAISS_DB_PATH, FAISS_INDEX_PATH]:
    if os.path.exists(p): os.remove(p)

# Create on disk -- pass a file path instead of :memory:
fdb_disk = FaissVectorDB(dim=768, path=FAISS_DB_PATH)
fdb_disk.add_batch(texts, embed_matrix, metas)
fdb_disk.save(FAISS_INDEX_PATH)

print(f"Created   : {fdb_disk}")
print(f"SQLite    : {os.path.getsize(FAISS_DB_PATH):,} bytes  (text + metadata)")
print(f"FAISS idx : {os.path.getsize(FAISS_INDEX_PATH):,} bytes  (vectors)")
print()

# Simulate a process restart -- delete the Python object
del fdb_disk
print("Python object deleted. Data lives on disk in two files.")
print()

# Reload from the two files
fdb_reloaded = FaissVectorDB.load(dim=768, db_path=FAISS_DB_PATH, index_path=FAISS_INDEX_PATH)
print(f"Reloaded  : {fdb_reloaded}")

# Search still works
q_vec   = get_embedding("How do cells produce energy?")
results = fdb_reloaded.search(q_vec, top_k=3)
print()
print("Search after reload:")
for text, score, _ in results:
    print(f"  {score:.4f}  {text[:65]}...")

Created   : FaissVectorDB(path='./knowledge_faiss.db', docs=50)
SQLite    : 20,480 bytes  (text + metadata)
FAISS idx : 153,645 bytes  (vectors)

Python object deleted. Data lives on disk in two files.

Reloaded  : FaissVectorDB(path='./knowledge_faiss.db', docs=50)

Search after reload:
  0.7870  Mitochondria generate ATP through cellular respiration, powering ...
  0.6199  Photosynthesis converts CO2 and water into glucose using sunlight...
  0.5563  Fermentation transforms sugars into alcohol or acids, preserving ...


Same results, same scores. The FaissVectorDB is a **drop-in replacement** -- same API, same answers.

But the internal search path is completely different:

| Step | VectorDB (brute force) | FaissVectorDB |
|------|----------------------|---------------|
| 1 | `SELECT * FROM documents` (all N rows) | `index.search(query, k)` (C++ SIMD) |
| 2 | Deserialize N BLOBs → NumPy | Get k indices + distances |
| 3 | Matrix multiply (N, 768) @ (768,) | `SELECT ... WHERE id IN (k ids)` |
| 4 | Sort all N scores | Already sorted by FAISS |

The brute-force version touches **every row on every query**. The FAISS-backed version touches only **k rows** from SQLite (just to fetch the text). At 50 docs you won't feel it. At 50,000 you will.

Let's measure the scaling difference:

In [ ]:
# Scale test: brute-force VectorDB vs FAISS-backed VectorDB
# Same synthetic vectors, increasing N

D        = 768
rng      = np.random.default_rng(42)
N_QUERIES = 20

print(f"{'N docs':>8}  {'Brute ms':>10}  {'FAISS-DB ms':>12}  {'Speedup':>8}")
print("-" * 50)

for N in [50, 500, 5_000, 50_000]:
    # Generate random vectors
    bench_vecs = rng.standard_normal((N, D)).astype(np.float32)
    q_vecs     = rng.standard_normal((N_QUERIES, D)).astype(np.float32)
    doc_texts  = [f"doc_{i}" for i in range(N)]

    # Brute-force VectorDB
    bf_db = VectorDB()
    bf_db.add_batch(doc_texts, bench_vecs)

    t0 = time.perf_counter()
    for q in q_vecs:
        bf_db.search(q, top_k=5)
    bf_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    # FAISS-backed VectorDB
    fi_db = FaissVectorDB(dim=D)
    fi_db.add_batch(doc_texts, bench_vecs)

    t0 = time.perf_counter()
    for q in q_vecs:
        fi_db.search(q, top_k=5)
    fi_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    speedup = bf_ms / fi_ms if fi_ms > 0 else float('inf')
    print(f"{N:>8,}  {bf_ms:>10.2f}  {fi_ms:>12.2f}  {speedup:>7.1f}x")

print()
print("The FAISS index eliminates the SQLite BLOB bottleneck.")
print("At 50K docs, brute force scans 50,000 rows per query.")
print("FAISS-backed DB asks FAISS for 5 indices, then fetches 5 rows.")

  N docs    Brute ms   FAISS-DB ms   Speedup
--------------------------------------------------
      50        0.18          0.04      4.2x
     500        1.16          0.04     26.3x
   5,000        9.75          0.24     40.4x
  50,000      108.01          2.73     39.6x

The FAISS index eliminates the SQLite BLOB bottleneck.
At 50K docs, brute force scans 50,000 rows per query.
FAISS-backed DB asks FAISS for 5 indices, then fetches 5 rows.


---
## Part 9 -- ChromaDB: A Vector Database

ChromaDB is an open-source vector database used widely in production. Internally it does exactly what we just built -- SQLite for metadata, a vector index for search -- but with more engineering:

- **HNSW** (Hierarchical Navigable Small World graph): approximate O(log N) nearest-neighbour instead of exact O(N). Our FaissVectorDB used `IndexFlatL2` which is still exact brute-force inside FAISS. HNSW trades a tiny bit of accuracy for massive speed.
- Full client/server architecture
- Richer metadata filter operators
- Automatic embedding (optional)

Same 50 docs, same queries -- let's add the fourth competitor:

In [ ]:
import chromadb

# Create an in-memory ChromaDB client
chroma_client = chromadb.EphemeralClient()

# Safe to re-run: delete the collection if it already exists from a previous run
try:
    chroma_client.delete_collection("knowledge_chroma")
    print("(deleted existing 'knowledge_chroma' collection — fresh start)")
except Exception:
    pass  # nothing to delete on first run

chroma_col = chroma_client.create_collection(
    "knowledge_chroma",
    metadata={"hnsw:space": "cosine"},
)

# Load the same 50 documents with the same embeddings from Part 4
chroma_col.add(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    embeddings=[v.tolist() for v in embeddings],
    metadatas=metas,
)
print(f"ChromaDB: {chroma_col.count()} documents loaded")
print()

# Same flow: query text → embed → search
query = "How do living cells produce energy?"
print(f'Query: "{query}"')
q_vec = get_embedding(query)
print(f"Embedded → shape {q_vec.shape}")
print()

results = chroma_col.query(query_embeddings=[q_vec.tolist()], n_results=3)
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    score = 1 - dist  # ChromaDB returns cosine distance; convert to similarity
    print(f"  [{score:.4f}] {doc[:80]}")

ChromaDB: 50 documents loaded

Query: "How do living cells produce energy?"
Embedded → shape (768,)

  [0.7588] Mitochondria generate ATP through cellular respiration, powering the cell.
  [0.5987] Photosynthesis converts CO2 and water into glucose using sunlight.
  [0.5330] Fermentation transforms sugars into alcohol or acids, preserving food and adding


In [ ]:
# ── Part 10: Four-Way Benchmark ──────────────────────────────────────────────
#
# All 4 systems have the same 50 real documents loaded.
# We query each system with the same 50 sentences (one per document).
#
# The full pipeline for every query is always:
#   query text → get_embedding(text) → search(embedding, top_k=5)
#
# We pre-computed the embeddings in Part 4, so here we only time the
# search step -- that's the part that differs between systems.

N_Q = len(embeddings)

print(f"Benchmarking {N_Q} queries across all 4 systems...")
print(f"Example queries:")
for t in texts[:3]:
    print(f'  "{t[:60]}..."')
print(f"  ... and {N_Q - 3} more\n")

# 1. Our brute-force VectorDB (SQLite BLOB + NumPy)
t0 = time.perf_counter()
for q in embeddings:
    db.search(q, top_k=5)
bf_ms = (time.perf_counter() - t0) / N_Q * 1000

# 2. FAISS standalone (no database, just the index)
t0 = time.perf_counter()
for q in normed_queries:
    faiss_index.search(q.reshape(1, -1), 5)
faiss_ms = (time.perf_counter() - t0) / N_Q * 1000

# 3. FaissVectorDB (FAISS index + SQLite text/metadata)
t0 = time.perf_counter()
for q in embeddings:
    fdb.search(q, top_k=5)
fdb_ms = (time.perf_counter() - t0) / N_Q * 1000

# 4. ChromaDB (HNSW + SQLite)
t0 = time.perf_counter()
for q in embeddings:
    chroma_col.query(query_embeddings=[q.tolist()], n_results=5)
chroma_ms = (time.perf_counter() - t0) / N_Q * 1000

print("=" * 75)
print("  BENCHMARK: 50 real documents, 50 queries, top_k=5")
print("=" * 75)
print(f"  {'Backend':<25} {'ms/query':>10}  {'QPM':>10}  Architecture")
print("-" * 75)
print(f"  {'1. VectorDB (brute)':<25} {bf_ms:>10.3f}  {60000/bf_ms:>10,.0f}  SQLite BLOB + NumPy cosine")
print(f"  {'2. FAISS alone':<25} {faiss_ms:>10.3f}  {60000/faiss_ms:>10,.0f}  C++ index only (no DB)")
print(f"  {'3. FaissVectorDB':<25} {fdb_ms:>10.3f}  {60000/fdb_ms:>10,.0f}  FAISS index + SQLite text")
print(f"  {'4. ChromaDB':<25} {chroma_ms:>10.3f}  {60000/chroma_ms:>10,.0f}  HNSW index + SQLite")
print("=" * 75)
print()
print("Two things look surprising here:")
print()
print("1. ChromaDB is SLOWER than our brute-force DB.")
print("   At 50 docs this is expected. Brute force is just one tiny matrix")
print("   multiply: (50, 768) @ (768,). ChromaDB's HNSW graph setup,")
print("   Python→C++ bridge, and list serialization cost more than just")
print("   scanning 50 vectors directly.")
print()
print("2. FAISS is fastest -- but only because N is tiny.")
print("   FAISS IndexFlatL2 is exact brute-force in C++. At 50 docs it")
print("   flies. But it's still O(N) -- it scans EVERY vector on every query.")
print("   As N grows, it will slow down linearly, just like our VectorDB.")
print()
print("The real question: what happens when N gets large?")

Benchmarking 50 queries across all 4 systems...
Example queries:
  "Photosynthesis converts CO2 and water into glucose using sun..."
  "DNA encodes genetic instructions using four bases: adenine, ..."
  "The speed of light in a vacuum is exactly 299,792,458 metres..."
  ... and 47 more

  BENCHMARK: 50 real documents, 50 queries, top_k=5
  Backend                     ms/query         QPM  Architecture
---------------------------------------------------------------------------
  1. VectorDB (brute)            0.123     488,507  SQLite BLOB + NumPy cosine
  2. FAISS alone                 0.005  11,216,715  C++ index only (no DB)
  3. FaissVectorDB               0.018   3,258,068  FAISS index + SQLite text
  4. ChromaDB                    0.883      67,922  HNSW index + SQLite

Two things look surprising here:

1. ChromaDB is SLOWER than our brute-force DB.
   At 50 docs this is expected. Brute force is just one tiny matrix
   multiply: (50, 768) @ (768,). ChromaDB's HNSW graph setup,
   

In [ ]:
# ── Scale Benchmark: 100 → 100,000 documents ────────────────────────────────
#
# Let's increase N and watch the crossover happen.
# We use synthetic random vectors here (no Ollama) to isolate search speed.
# The math is the same -- cosine similarity doesn't care if vectors are real
# or random. We just need enough vectors to stress each system.

D         = 768
rng       = np.random.default_rng(42)
N_QUERIES = 20

print("=" * 80)
print("  SCALE BENCHMARK: how each system handles growing data")
print("  (synthetic 768-D vectors, 20 queries per size, top_k=5)")
print("=" * 80)
print(f"  {'N docs':>8}  {'Brute ms':>10}  {'FAISS ms':>10}  {'F+DB ms':>10}  {'Chroma ms':>10}")
print("-" * 80)

for N in [100, 1_000, 10_000, 50_000, 100_000]:
    vecs   = rng.standard_normal((N, D)).astype(np.float32)
    q_vecs = rng.standard_normal((N_QUERIES, D)).astype(np.float32)
    labels = [f"doc_{i}" for i in range(N)]

    # --- 1. Brute-force VectorDB ---
    bf = VectorDB()
    bf.add_batch(labels, vecs)
    t0 = time.perf_counter()
    for q in q_vecs:
        bf.search(q, top_k=5)
    bf_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    # --- 2. FAISS standalone (IndexFlatL2 = exact brute-force in C++) ---
    fi = faiss.IndexFlatL2(D)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-10
    fi.add(vecs / norms)
    q_normed = q_vecs / (np.linalg.norm(q_vecs, axis=1, keepdims=True) + 1e-10)
    t0 = time.perf_counter()
    for q in q_normed:
        fi.search(q.reshape(1, -1), 5)
    fi_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    # --- 3. FaissVectorDB (same FAISS index + SQLite text lookup) ---
    fd = FaissVectorDB(dim=D)
    fd.add_batch(labels, vecs)
    t0 = time.perf_counter()
    for q in q_vecs:
        fd.search(q, top_k=5)
    fd_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    # --- 4. ChromaDB (HNSW = approximate, O(log N)) ---
    try:
        chroma_client.delete_collection("scale_bench")
    except Exception:
        pass
    col = chroma_client.create_collection("scale_bench", metadata={"hnsw:space": "cosine"})
    batch_size = 5000
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        col.add(
            ids=[str(i) for i in range(start, end)],
            documents=labels[start:end],
            embeddings=[v.tolist() for v in vecs[start:end]],
        )
    t0 = time.perf_counter()
    for q in q_vecs:
        col.query(query_embeddings=[q.tolist()], n_results=5)
    ch_ms = (time.perf_counter() - t0) / N_QUERIES * 1000

    print(f"  {N:>8,}  {bf_ms:>10.2f}  {fi_ms:>10.2f}  {fd_ms:>10.2f}  {ch_ms:>10.2f}")

    chroma_client.delete_collection("scale_bench")

print("=" * 80)
print()
print("Three things to notice:")
print()
print("1. ChromaDB WINS at large N, even though it was slowest at 50 docs.")
print("   Its HNSW index is O(log N) -- search barely slows down as N grows.")
print("   Our brute-force DB is O(N) -- it doubles every time N doubles.")
print()
print("2. FAISS alone and FaissVectorDB are nearly IDENTICAL.")
print("   That means wrapping FAISS in SQLite adds almost zero overhead.")
print("   The 5 primary-key lookups (to fetch text) cost microseconds.")
print("   This is the proof: you can have both speed AND a real database.")
print()
print("3. FAISS (IndexFlatL2) is ALSO slower than ChromaDB at 100K.")
print("   Why? Because IndexFlatL2 is exact brute-force -- it scans every")
print("   vector, just in C++ instead of Python. Still O(N).")
print("   ChromaDB's HNSW skips most comparisons entirely -- O(log N).")
print()
print("The lesson: the ALGORITHM matters more than the language.")
print("  Python brute-force → C++ brute-force: ~20x faster (same O(N))")
print("  C++ brute-force → HNSW approximate: fundamentally different scaling")

  SCALE BENCHMARK: how each system handles growing data
  (synthetic 768-D vectors, 20 queries per size, top_k=5)
    N docs    Brute ms    FAISS ms     F+DB ms   Chroma ms
--------------------------------------------------------------------------------
       100        0.20        0.01        0.02        0.51
     1,000        1.86        0.04        0.06        0.74
    10,000       21.33        0.48        0.50        0.83
    50,000      110.23        2.49        2.58        0.80
   100,000      231.61        4.47        4.54        0.90

Three things to notice:

1. ChromaDB WINS at large N, even though it was slowest at 50 docs.
   Its HNSW index is O(log N) -- search barely slows down as N grows.
   Our brute-force DB is O(N) -- it doubles every time N doubles.

2. FAISS alone and FaissVectorDB are nearly IDENTICAL.
   That means wrapping FAISS in SQLite adds almost zero overhead.
   The 5 primary-key lookups (to fetch text) cost microseconds.
   This is the proof: you can have 

In [ ]:
# Do all four agree on the top result?

print(f"  {'Query':<45}  {'Brute':>5}  {'FAISS':>5}  {'F+DB':>5}  {'Chroma':>6}")
print("-" * 80)
for q_text in VERIFY_QUERIES:
    q_vec  = get_embedding(q_text)
    q_norm = q_vec / np.linalg.norm(q_vec)

    bf_res  = db.search(q_vec, top_k=1)[0][0]
    _, I    = faiss_index.search(q_norm.reshape(1, -1), 1)
    f_res   = texts[I[0][0]]
    fdb_res = fdb.search(q_vec, top_k=1)[0][0]
    c_res   = chroma_col.query(query_embeddings=[q_vec.tolist()], n_results=1)["documents"][0][0]

    # Compare all against ChromaDB as ground truth
    bf_ok  = "OK" if bf_res  == c_res else "NO"
    f_ok   = "OK" if f_res   == c_res else "NO"
    fdb_ok = "OK" if fdb_res == c_res else "NO"
    print(f"  {q_text[:45]:<45}  {bf_ok:>5}  {f_ok:>5}  {fdb_ok:>5}  {'ref':>6}")

print()
print("ChromaDB used as ground truth (HNSW with cosine space).")
print("All four implementations return the same top result -- same math, different speed.")

  Query                                          Brute  FAISS   F+DB  Chroma
--------------------------------------------------------------------------------


  How do living cells produce energy?               OK     OK     OK     ref
  What caused ancient civilisations to collapse     OK     OK     OK     ref
  What makes bread dough ferment?                   OK     OK     OK     ref
  How do neural networks learn?                     OK     OK     OK     ref
  Which events test all-round athletic ability?     OK     OK     OK     ref

ChromaDB used as ground truth (HNSW with cosine space).
All four implementations return the same top result -- same math, different speed.


---
## The Crux

A vector database is three things stacked:

**1. An embedding model** converts text to numbers. We used Ollama's `nomic-embed-text`. 768 floats per sentence.

**2. A similarity function** compares two sets of numbers. We used cosine similarity. One dot product and two norms.

**3. A storage layer** holds the text and numbers, retrieves them fast. We used SQLite with vectors as BLOBs.

We built four versions, each teaching one lesson:

```
Version 1: VectorDB (brute-force)
┌─────────────────────────────────────────────┐
│  SQLite stores EVERYTHING (text + vectors)  │
│  Search = load ALL vectors + NumPy cosine   │
│  Simple. Correct. Slow at scale. O(N).      │
└─────────────────────────────────────────────┘
        │
        │  "What if we skip the database for vectors?"
        ▼
Version 2: FAISS alone (IndexFlatL2)
┌─────────────────────────────────────────────┐
│  Vectors in RAM, C++ SIMD dot products      │
│  Still O(N) -- same algorithm, faster lang  │
│  No text, no metadata, no disk.             │
└─────────────────────────────────────────────┘
        │
        │  "What if we combine them?"
        ▼
Version 3: FaissVectorDB
┌─────────────────────────────────────────────┐
│  FAISS handles vectors (fast search)        │
│  SQLite handles text + metadata (storage)   │
│  Each tool does what it's best at.          │
│  Nearly zero overhead vs FAISS alone.       │
└─────────────────────────────────────────────┘
        │
        │  "What does a production DB add on top?"
        ▼
Version 4: ChromaDB (HNSW)
┌─────────────────────────────────────────────┐
│  HNSW graph: O(log N) approximate search    │
│  Different algorithm, not just faster code  │
│  Slower at 50 docs. Fastest at 100K.        │
└─────────────────────────────────────────────┘
```

The benchmark revealed something important: **the algorithm matters more than the implementation language.**

- Python brute-force → C++ brute-force (FAISS): ~20x faster, but still O(N). Both scale linearly.
- C++ brute-force → HNSW (ChromaDB): fundamentally different scaling. O(N) vs O(log N).
- Wrapping FAISS in SQLite (FaissVectorDB): nearly zero overhead. You can have speed AND a real database.

When someone says "we use Pinecone" or "we switched to Qdrant" -- now you know exactly what is happening under the hood. The embedding model produces numbers. The index finds the closest ones. The database stores and retrieves the text. The index algorithm determines how it scales.

**Now you have seen every line of it.**